# Memora AI — Cognitive Performance ML Training

Train a model from game-session features for **cognitive-performance pattern monitoring** and adaptive difficulty.

⚠️ If no approved dataset is supplied, the notebook creates synthetic development data. This is NOT clinical validation and must not be presented as a dementia diagnostic model.


In [ ]:
!pip -q install joblib scikit-learn pandas matplotlib

import os, json, joblib, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

SEED = 42
np.random.seed(SEED)
ARTIFACT_DIR = "ml_colab/artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)


In [ ]:
FEATURES = ['memory_journey_score', 'memory_matrix_score', 'balloon_score', 'memory_journey_time', 'memory_matrix_time', 'matrix_moves', 'balloon_max_sequence', 'hand_detections', 'face_frames', 'head_movement', 'overall_score', 'accuracy', 'reaction_efficiency', 'memory_consistency', 'session_score_change', 'session_time_change', 'difficulty_level']
TARGET = "performance_pattern"

with open(f"{ARTIFACT_DIR}/feature_schema.json", "w") as f:
    json.dump({"features": FEATURES, "target": TARGET}, f, indent=2)

print("Training features:", len(FEATURES))


## Load your approved dataset

Set `DATA_PATH` to a CSV uploaded to Colab. The CSV must contain the feature columns and `performance_pattern`.

If you leave it empty, a transparent synthetic development dataset is generated only to test the ML pipeline.


In [ ]:
DATA_PATH = ""  # e.g. "/content/approved_dataset.csv"

if DATA_PATH and os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    DATA_SOURCE = "real_approved_dataset"
else:
    DATA_SOURCE = "synthetic_development_data"
    n = 3000
    rng = np.random.default_rng(SEED)

    df = pd.DataFrame({
        "memory_journey_score": rng.normal(70,16,n).clip(0,100),
        "memory_matrix_score": rng.normal(68,17,n).clip(0,100),
        "balloon_score": rng.normal(72,15,n).clip(0,100),
        "memory_journey_time": rng.normal(55,14,n).clip(15,120),
        "memory_matrix_time": rng.normal(65,16,n).clip(15,140),
        "matrix_moves": rng.normal(28,9,n).clip(5,80),
        "balloon_max_sequence": rng.normal(8,2.5,n).clip(1,15),
        "hand_detections": rng.normal(35,10,n).clip(0,100),
        "face_frames": rng.normal(80,12,n).clip(0,100),
        "head_movement": rng.normal(40,15,n).clip(0,100),
        "difficulty_level": rng.integers(1,4,n)
    })

    df["overall_score"] = (
        .30*df.memory_journey_score + .30*df.memory_matrix_score +
        .25*df.balloon_score + .15*(df.balloon_max_sequence/15)*100
    ).clip(0,100)
    df["accuracy"] = (
        (df.memory_journey_score + df.memory_matrix_score + df.balloon_score)/3
    ).clip(0,100)
    df["reaction_efficiency"] = (
        100 - (df.memory_journey_time + df.memory_matrix_time)/2
    ).clip(0,100)
    df["memory_consistency"] = (
        100 - abs(df.memory_journey_score - df.memory_matrix_score)
    ).clip(0,100)
    df["session_score_change"] = rng.normal(0,8,n).clip(-30,30)
    df["session_time_change"] = rng.normal(0,10,n).clip(-40,40)

    risk_score = (
        .45*df.overall_score + .20*df.accuracy +
        .15*df.memory_consistency + .10*df.reaction_efficiency +
        .10*(50 + df.session_score_change)
    )
    df["performance_pattern"] = pd.cut(
        risk_score,
        bins=[-np.inf,55,72,np.inf],
        labels=["higher_concern_pattern","monitor_pattern","lower_concern_pattern"]
    ).astype(str)

print(DATA_SOURCE, df.shape)
display(df.head())


In [ ]:
missing = [c for c in FEATURES + [TARGET] if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df[FEATURES + [TARGET]].copy()
print(df[TARGET].value_counts())


## Train and compare models

In [ ]:
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.20, random_state=SEED, stratify=y
)

models = {
    "logistic_regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=SEED))
    ]),
    "random_forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=400, random_state=SEED, class_weight="balanced"
        ))
    ]),
    "gradient_boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingClassifier(random_state=SEED))
    ])
}

results, trained = [], {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    p,r,f1,_ = precision_recall_fscore_support(
        y_test, pred, average="weighted", zero_division=0
    )
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test,pred),
        "precision_weighted": p,
        "recall_weighted": r,
        "f1_weighted": f1
    })
    trained[name] = model

results_df = pd.DataFrame(results).sort_values("f1_weighted", ascending=False)
display(results_df)


## Evaluate the selected model

In [ ]:
best_name = results_df.iloc[0]["model"]
best_model = trained[best_name]
best_pred = best_model.predict(X_test)

print("Selected model:", best_name)
print(classification_report(y_test, best_pred, zero_division=0))

cm = confusion_matrix(y_test, best_pred, labels=sorted(y.unique()))
plt.figure(figsize=(7,5))
plt.imshow(cm)
plt.title(f"Confusion Matrix — {best_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(range(len(sorted(y.unique()))), sorted(y.unique()), rotation=30, ha="right")
plt.yticks(range(len(sorted(y.unique()))), sorted(y.unique()))
plt.tight_layout()
plt.show()


## Save artifacts for Flask

In [ ]:
joblib.dump(best_model, f"{ARTIFACT_DIR}/memora_model.joblib")

metrics = {
    "selected_model": best_name,
    "accuracy": float(results_df.iloc[0].accuracy),
    "precision_weighted": float(results_df.iloc[0].precision_weighted),
    "recall_weighted": float(results_df.iloc[0].recall_weighted),
    "f1_weighted": float(results_df.iloc[0].f1_weighted),
    "data_source": DATA_SOURCE,
    "warning": "Prototype cognitive-performance model; not a clinical diagnostic model."
}
with open(f"{ARTIFACT_DIR}/model_metrics.json","w") as f:
    json.dump(metrics,f,indent=2)

print("Artifacts saved:")
for name in os.listdir(ARTIFACT_DIR):
    print("-", name)


## Test one prediction

In [ ]:
sample = X_test.iloc[[0]]
print("Predicted pattern:", best_model.predict(sample)[0])
if hasattr(best_model, "predict_proba"):
    probs = best_model.predict_proba(sample)[0]
    print(dict(zip(best_model.classes_, map(float, probs))))
